# 08 펌프 센서 시계열 피처: rolling 통계와 변화율 분포

1. 입력 준비 — 전방 채움, 고장 시작 시각, 분석용 상태 구분 (기획서의 기간 분할은 이 단계에서 적용하지 않음)
2. 변화율 분포 — 1·5·15·60분 변화량을 정상·고장 직전·회복 구간별로 비교
3. rolling 통계 — 15·60·360분 창의 평균·표준편차·최소·최대와 360분 z-score
4. 고장 전조 점검 — 고장 7건의 직전 1시간을 24~6시간 전 기준과 비교
5. 단일 센서 규칙의 선행 시간과 오경보
6. 피처 저장 — `data/processed/pump_features.parquet`, `docs/feature_spec.json`

모든 피처는 **해당 시각 이전 값만** 사용합니다. 결측은 직전 관측값으로 채우고(전방 채움), rolling 창은
현재 시각에서 끝나며, 변화량은 과거 값과의 차이입니다. EDA에서 쓴 전체 기간 선형보간과 달리 결측
뒤의 값을 쓰지 않으므로 예측 모델 입력으로 바로 쓸 수 있습니다. 계산 코드는 `src/pump_features.py`에
있습니다.

## 1. 입력 준비

`sensor_15`, `sensor_50`, `sensor_51`을 제외한 49개 센서를 읽고 전방 채움합니다. 원본 `machine_status`의
`BROKEN` 행이 고장 시작 시각입니다. 분석용 상태 `state`는 고장 시작 전 60분의 NORMAL 행을
`pre_failure`로 따로 표시합니다.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
project_root = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "src" / "pump_features.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("kkobugi 저장소 또는 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(project_root))

from src.pump_missingness_audit import MINUTE
from src.pump_features import (
    DIFF_LAGS, WINDOWS, ZSCORE_WINDOW, analysis_state, build_features, diff_features,
    failure_starts, feature_spec, load_prepared, rolling_features, zscore_features,
)

PRE_FAILURE_MINUTES = 60
figure_dir = project_root / "figures/pump_features"
figure_dir.mkdir(parents=True, exist_ok=True)

prepared = load_prepared()
values, status, label = (prepared[key] for key in ("values", "status", "label"))
sensors = list(values.columns)
starts = failure_starts(status)
state = analysis_state(status, starts, PRE_FAILURE_MINUTES)

print(f"원본 {prepared['source'].name} · {len(values):,}행 · 센서 {len(sensors)}개 · "
      f"전방 채움 {int(prepared['missing'].to_numpy().sum()):,}개")
failures = pd.DataFrame({
    "고장 시작": starts,
    "회복 종료": [
        status.index[(status.index > start) & status.eq("NORMAL")].min() for start in starts
    ],
})
failures["회복 시간(분)"] = ((failures["회복 종료"] - failures["고장 시작"]) // MINUTE).astype(int)
display(failures)
display(state.value_counts().rename("행 수").to_frame().T)

원본 sensor.csv · 220,320행 · 센서 49개 · 전방 채움 31,854개


,고장 시작,회복 종료,회복 시간(분)
0,2018-04-12 21:55:00,2018-04-13 13:40:00,945
1,2018-04-18 00:30:00,2018-04-20 04:21:00,3111
2,2018-05-19 03:18:00,2018-05-20 01:11:00,1313
3,2018-05-25 00:30:00,2018-05-25 10:36:00,606
4,2018-06-28 22:00:00,2018-07-04 17:51:00,8391
5,2018-07-08 00:11:00,2018-07-08 00:53:00,42
6,2018-07-25 14:00:00,2018-07-25 15:16:00,76


state,normal,recovering,pre_failure,broken
행 수,205416,14477,420,7


## 2. 변화율 분포

변화량 `diff{lag}` = 현재 값 − lag분 전 값입니다. 센서마다 단위가 달라 센서별로 비교합니다.

- **정상 기준 범위**: 전체 기간의 `normal` 행(고장 직전 60분 제외)에서 변화량의 1%·99% 분위수를
  구합니다. 탐색용 기준이며, 모델 검증 때는 학습 구간에서만 다시 구해야 합니다.
- **이탈 비율**: 상태별로 변화량이 정상 기준 범위를 벗어난 행의 비율입니다. 정상 구간의 이탈 비율은
  약 2%가 기대값이고, `pre_failure`에서 그보다 크게 높으면 고장 직전에 움직임이 커진 센서입니다.

In [ ]:
diffs = diff_features(values, DIFF_LAGS, dtype="float64")
is_normal = state.eq("normal")

rows = []
for lag in DIFF_LAGS:
    columns = [f"{sensor}__diff{lag}" for sensor in sensors]
    frame = diffs[columns]
    low = frame[is_normal].quantile(0.01)
    high = frame[is_normal].quantile(0.99)
    outside = frame.lt(low) | frame.gt(high)
    for name in ("normal", "pre_failure", "recovering"):
        share = outside[state.eq(name)].mean()
        for sensor, column in zip(sensors, columns):
            rows.append({"센서": sensor, "lag": lag, "상태": name, "이탈 비율(%)": 100 * share[column]})
escape = (
    pd.DataFrame(rows)
    .pivot_table(index=["센서", "lag"], columns="상태", values="이탈 비율(%)")
    [["normal", "pre_failure", "recovering"]]
)
escape["pre_failure − normal"] = escape["pre_failure"] - escape["normal"]
print("고장 직전 60분의 변화량 이탈 비율이 정상 대비 가장 높은 센서·lag (상위 15)")
display(escape.sort_values("pre_failure − normal", ascending=False).head(15).round(1))

고장 직전 60분의 변화량 이탈 비율이 정상 대비 가장 높은 센서·lag (상위 15)


상태             normal  pre_failure  recovering  pre_failure − normal
센서        lag                                                       
sensor_04 60      2.0         23.8         8.7                  21.8
sensor_00 15      2.0         19.5         9.6                  17.5
          60      2.0         17.9         8.4                  15.9
sensor_12 60      2.0         17.9         2.5                  15.9
sensor_04 15      2.0         17.6         3.1                  15.6
sensor_13 60      2.0         14.0         0.9                  12.0
sensor_00 5       2.0         13.8         8.5                  11.8
sensor_04 5       2.0         13.8         1.4                  11.8
sensor_34 60      2.0         13.8         1.5                  11.8
sensor_35 60      2.0         13.8         1.9                  11.8
sensor_08 60      2.0         13.6        13.7                  11.6
sensor_36 60      2.0         13.6         3.1                  11.6
sensor_09 60      2.0         12.9        15.1                  10.9
sensor_26 5       2.0         12.9         2.7                  10.9
sensor_32 15      2.0         11.7         2.8                   9.7

In [ ]:
candidates = ["sensor_04", "sensor_03", "sensor_48", "sensor_19"]
lag = 15
quantiles = [0.01, 0.05, 0.5, 0.95, 0.99]
table = pd.concat(
    {
        name: diffs.loc[state.eq(name), [f"{sensor}__diff{lag}" for sensor in candidates]]
        .quantile(quantiles).T.set_axis(candidates)
        for name in ("normal", "pre_failure", "recovering")
    },
    axis=1,
)
print(f"후보 센서의 {lag}분 변화량 분위수 (상태별)")
display(table.round(3))

fig, axes = plt.subplots(1, len(candidates), figsize=(4.2 * len(candidates), 3.6))
for ax, sensor in zip(axes, candidates):
    column = f"{sensor}__diff{lag}"
    normal_values = diffs.loc[state.eq("normal"), column].dropna()
    pre_values = diffs.loc[state.eq("pre_failure"), column].dropna()
    limit = np.nanpercentile(np.abs(normal_values), 99.9)
    bins = np.linspace(-limit, limit, 61)
    ax.hist(normal_values.clip(-limit, limit), bins=bins, density=True, alpha=.55,
            color="#64748b", label="normal")
    ax.hist(pre_values.clip(-limit, limit), bins=bins, density=True, alpha=.65,
            color="#dc2626", label=f"pre-failure ({PRE_FAILURE_MINUTES} min)")
    ax.set_yscale("log")
    ax.set_title(f"{sensor} diff{lag}")
    ax.set_xlabel(f"value − value {lag} min earlier")
axes[0].set_ylabel("density (log)")
axes[0].legend(fontsize=8)
fig.suptitle("Change-rate distribution: normal vs 60 min before failure", y=1.02)
fig.tight_layout()
fig.savefig(figure_dir / f"diff{lag}_distribution_candidates.png", dpi=150, bbox_inches="tight")
plt.show()

후보 센서의 15분 변화량 분위수 (상태별)


normal                                pre_failure                  \
             0.01    0.05   0.50    0.95    0.99        0.01     0.05   0.50   
sensor_04 -20.255 -13.657  0.116  13.773  21.181    -499.935 -266.695 -2.083   
sensor_03  -0.434  -0.260  0.000   0.217   0.304      -0.347   -0.260 -0.087   
sensor_48 -88.542 -53.820 -0.289  54.977  90.567     -35.246  -26.071  4.340   
sensor_19 -13.062  -4.706  0.000   5.037  19.566    -127.171   -8.156  0.184   

                           recovering                             
             0.95     0.99       0.01   0.05 0.50   0.95    0.99  
sensor_04  12.153   17.708    -82.056 -3.125  0.0  2.894  60.424  
sensor_03   0.217    0.296     -0.781 -0.434  0.0  0.347   0.998  
sensor_48  63.672  115.729    -40.289 -3.472  0.0  0.868  28.715  
sensor_19   5.197   32.538    -13.689 -4.273 -0.0  4.502  35.824

## 3. rolling 통계

창 15·60·360분의 평균·표준편차·최소·최대와, 직전 360분 기준 z-score를 계산합니다. 창이 다 차기 전
(시계열 맨 앞 window−1행)은 NaN으로 둡니다. 아래 그림은 고장 7건 각각에 대해 고장 24시간 전부터
6시간 후까지 `sensor_04`의 원본 값, 60분 rolling 표준편차, 360분 z-score입니다.

In [ ]:
features = build_features(values, WINDOWS, lags=DIFF_LAGS, zscore_window=ZSCORE_WINDOW).copy()
memory_mb = features.memory_usage(deep=True).sum() / 2**20
print(f"피처 {features.shape[1]:,}개 × {len(features):,}행 · 메모리 {memory_mb:,.0f} MB")

focus = "sensor_04"
panels = [(focus, "raw"), (f"{focus}__roll60_std", "rolling std (60 min)"),
          (f"{focus}__z{ZSCORE_WINDOW}", f"z-score ({ZSCORE_WINDOW} min)")]
fig, axes = plt.subplots(len(panels), len(starts), figsize=(3.3 * len(starts), 7.5), sharex="col")
for col, start in enumerate(starts):
    window = features.loc[start - pd.Timedelta(hours=24): start + pd.Timedelta(hours=6)]
    hours = (window.index - start) / pd.Timedelta(hours=1)
    for row, (column, title) in enumerate(panels):
        ax = axes[row, col]
        ax.plot(hours, window[column], lw=.8, color="#1d4ed8")
        ax.axvline(0, color="#dc2626", lw=1)
        ax.axvspan(-PRE_FAILURE_MINUTES / 60, 0, color="#fecaca", alpha=.5)
        if row == 0:
            ax.set_title(start.strftime("%m-%d %H:%M"), fontsize=10)
        if col == 0:
            ax.set_ylabel(title, fontsize=9)
    axes[-1, col].set_xlabel("hours from failure")
fig.suptitle(f"{focus} around each failure (red band = last {PRE_FAILURE_MINUTES} min)", y=1.01)
fig.tight_layout()
fig.savefig(figure_dir / f"{focus}_rolling_around_failures.png", dpi=150, bbox_inches="tight")
plt.show()

피처 882개 × 220,320행 · 메모리 743 MB


## 4. 고장 전조 점검

고장마다 **기준 구간**(24~6시간 전)과 **직전 구간**(1시간 전~고장 직전)을 비교합니다.

- `레벨 점수` = (직전 구간 평균 − 기준 구간 평균) / 기준 구간 표준편차. 절댓값 3 이상이면 평소 변동
  폭을 크게 벗어난 것입니다.
- `변동 비율` = 직전 구간 60분 rolling 표준편차 중앙값 / 기준 구간 60분 rolling 표준편차 중앙값.
  3 이상이면 직전에 출렁임이 3배 이상 커진 것입니다.

고장 7건 중 몇 건에서 기준을 넘었는지 세어 센서를 정렬합니다. 인수인계 EDA(`sensor_03`·`sensor_48`
5/7건 하락, 대조 구간 반영 시 `sensor_04` 3/7건)와 비교합니다.

In [ ]:
LEVEL_THRESHOLD, VOLATILITY_THRESHOLD = 3.0, 3.0
level_rows, volatility_rows = [], []
for start in starts:
    baseline = slice(start - pd.Timedelta(hours=24), start - pd.Timedelta(hours=6))
    recent = slice(start - pd.Timedelta(minutes=PRE_FAILURE_MINUTES), start - MINUTE)
    base_values, recent_values = values.loc[baseline], values.loc[recent]
    base_std = base_values.std().replace(0, np.nan)
    level_rows.append(((recent_values.mean() - base_values.mean()) / base_std).rename(start))

    std_columns = [f"{sensor}__roll60_std" for sensor in sensors]
    base_std60 = features.loc[baseline, std_columns].median().set_axis(sensors).replace(0, np.nan)
    recent_std60 = features.loc[recent, std_columns].median().set_axis(sensors)
    volatility_rows.append((recent_std60 / base_std60).rename(start))

level = pd.concat(level_rows, axis=1).astype("float64")
volatility = pd.concat(volatility_rows, axis=1).astype("float64")

precursor = pd.DataFrame({
    "레벨 하락 건수": level.le(-LEVEL_THRESHOLD).sum(axis=1),
    "레벨 상승 건수": level.ge(LEVEL_THRESHOLD).sum(axis=1),
    "변동 증가 건수": volatility.ge(VOLATILITY_THRESHOLD).sum(axis=1),
    "레벨 점수 중앙값": level.median(axis=1),
    "변동 비율 중앙값": volatility.median(axis=1),
})
precursor["전조 건수(레벨 또는 변동)"] = (
    level.abs().ge(LEVEL_THRESHOLD) | volatility.ge(VOLATILITY_THRESHOLD)
).sum(axis=1)
precursor = precursor.sort_values(
    ["전조 건수(레벨 또는 변동)", "변동 증가 건수"], ascending=False
)
print(f"고장 7건 중 직전 {PRE_FAILURE_MINUTES}분에 전조가 나타난 건수 (상위 15 센서)")
display(precursor.head(15).round(2))

print("후보 센서의 고장별 레벨 점수 (음수 = 평소보다 하락)")
display(level.loc[candidates].T.round(2).set_axis([s.strftime("%m-%d %H:%M") for s in starts]))
print("후보 센서의 고장별 변동 비율 (직전 60분 rolling std / 기준)")
display(volatility.loc[candidates].T.round(2).set_axis([s.strftime("%m-%d %H:%M") for s in starts]))

고장 7건 중 직전 60분에 전조가 나타난 건수 (상위 15 센서)


,레벨 하락 건수,레벨 상승 건수,변동 증가 건수,레벨 점수 중앙값,변동 비율 중앙값,전조 건수(레벨 또는 변동)
sensor_00,4,0,3,-3.84,1.16,5
sensor_06,3,0,1,-1.73,1.02,4
sensor_04,4,0,0,-13.83,1.07,4
sensor_13,0,2,3,0.63,1.70,3
sensor_43,0,1,2,-0.34,0.89,3
sensor_07,1,0,1,-0.23,0.90,2
sensor_09,1,1,1,-0.15,1.06,2
sensor_12,1,0,1,0.23,0.95,2
sensor_40,0,1,1,-0.32,0.96,2
sensor_02,2,0,0,-0.62,1.18,2


후보 센서의 고장별 레벨 점수 (음수 = 평소보다 하락)


,sensor_04,sensor_03,sensor_48,sensor_19
04-12 21:55,-15.57,-1.44,1.54,1.66
04-18 00:30,-17.24,-0.67,-0.80,1.06
05-19 03:18,-1.65,-1.41,-1.02,0.06
05-25 00:30,-0.09,-1.08,3.65,0.02
06-28 22:00,-13.83,-3.26,-1.75,2.05
07-08 00:11,-1.34,2.21,-0.70,0.03
07-25 14:00,-14.08,-0.52,-0.50,-0.04


후보 센서의 고장별 변동 비율 (직전 60분 rolling std / 기준)


,sensor_04,sensor_03,sensor_48,sensor_19
04-12 21:55,1.12,2.39,1.04,1.04
04-18 00:30,1.72,1.33,0.56,1.01
05-19 03:18,1.06,1.56,0.57,0.92
05-25 00:30,1.07,1.87,2.20,0.71
06-28 22:00,0.41,1.42,0.02,0.75
07-08 00:11,0.97,0.39,0.69,0.60
07-25 14:00,2.55,1.30,0.63,1.18


## 5. 단일 센서 규칙의 선행 시간과 오경보

가장 단순한 기준 규칙으로 "360분 z-score가 임계값을 넘으면 경보"를 시험합니다. 정상 행에서
z-score가 임계값을 넘는 비율이 낮으면서 고장 직전에는 넘는 센서가 유용합니다.
고장별로 **고장 전 6시간 안에서 처음 경보가 울린 시각**까지의 선행 시간(분)과, 고장 직전 6시간과
회복 구간을 제외한 정상 구간에서 울린 **오경보 비율**과, 연속된 경보를 한 번으로 센 **오경보 에피소드
수**를 함께 봅니다. 모델이 아니라 피처의 쓸모를
보는 기준선입니다.

In [ ]:
Z_THRESHOLD = 4.0
LEAD_HOURS = 6
z_columns = [f"{sensor}__z{ZSCORE_WINDOW}" for sensor in sensors]
z = features[z_columns].set_axis(sensors, axis=1).astype("float64")
alarm = z.abs().ge(Z_THRESHOLD)

# 고장 전 6시간은 '정상이지만 전조일 수 있는 구간'이므로 오경보 계산에서 제외합니다.
pre_window = pd.Series(False, index=values.index)
for start in starts:
    pre_window.loc[start - pd.Timedelta(hours=LEAD_HOURS): start - MINUTE] = True
clean_normal = state.eq("normal") & ~pre_window

lead_rows = []
for start in starts:
    window = alarm.loc[start - pd.Timedelta(hours=LEAD_HOURS): start - MINUTE]
    first = window.idxmax().where(window.any())
    lead_rows.append(((start - first) / MINUTE).rename(start.strftime("%m-%d %H:%M")))
lead = pd.concat(lead_rows, axis=1)

# 정상 구간에서 경보가 새로 켜진 횟수(연속 경보는 한 에피소드)를 셉니다.
clean_alarm = alarm & clean_normal.to_numpy()[:, None]
episodes = (clean_alarm & ~clean_alarm.shift(fill_value=False)).sum()

rule = pd.DataFrame({
    "적중 건수(7건 중)": lead.notna().sum(axis=1),
    "선행 시간 중앙값(분)": lead.median(axis=1),
    "정상 구간 오경보(%)": 100 * alarm[clean_normal].mean(),
    "오경보 에피소드 수": episodes,
}).sort_values(["적중 건수(7건 중)", "정상 구간 오경보(%)"], ascending=[False, True])
print(f"|z{ZSCORE_WINDOW}| ≥ {Z_THRESHOLD} 규칙 · 고장 전 {LEAD_HOURS}시간 안 첫 경보 기준 (상위 15 센서)")
display(rule.head(15).round(2))
print("후보 센서의 고장별 선행 시간(분, 비어 있으면 6시간 안 경보 없음)")
display(lead.loc[candidates])

|z360| ≥ 4.0 규칙 · 고장 전 6시간 안 첫 경보 기준 (상위 15 센서)


,적중 건수(7건 중),선행 시간 중앙값(분),정상 구간 오경보(%),오경보 에피소드 수
sensor_04,7,21.0,0.13,93
sensor_12,6,175.5,0.53,130
sensor_14,6,208.0,0.82,476
sensor_00,6,26.0,1.06,483
sensor_06,6,80.0,1.07,750
sensor_07,6,121.5,1.40,749
sensor_02,5,151.0,0.21,272
sensor_10,5,14.0,0.31,164
sensor_27,5,73.0,0.54,625
sensor_19,5,151.0,0.61,522


후보 센서의 고장별 선행 시간(분, 비어 있으면 6시간 안 경보 없음)


,04-12 21:55,04-18 00:30,05-19 03:18,05-25 00:30,06-28 22:00,07-08 00:11,07-25 14:00
sensor_04,21.0,48.0,2.0,4.0,360.0,6.0,52.0
sensor_03,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sensor_48,4.0,NaN,229.0,297.0,NaN,NaN,NaN
sensor_19,151.0,249.0,311.0,NaN,15.0,NaN,99.0


In [ ]:
# 적중 건수 대비 오경보가 낮은 센서를 한눈에 봅니다.
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(rule["정상 구간 오경보(%)"], rule["적중 건수(7건 중)"], s=28, color="#1d4ed8", alpha=.75)
for sensor, row in rule.iterrows():
    if row["적중 건수(7건 중)"] >= 4 or sensor in candidates:
        ax.annotate(sensor.replace("sensor_", "s"), (row["정상 구간 오경보(%)"], row["적중 건수(7건 중)"]),
                    fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("false alarm rate in clean normal rows (%)")
ax.set_ylabel(f"failures with alarm in last {LEAD_HOURS} h (of 7)")
ax.set_title(f"Single-sensor rule |z{ZSCORE_WINDOW}| ≥ {Z_THRESHOLD}")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(figure_dir / f"zscore_rule_hits_vs_false_alarms.png", dpi=150)
plt.show()

## 6. 피처 저장

피처 전체를 float32 parquet로 저장합니다(`data/`는 `.gitignore`에 있으므로 팀원은 이 노트북을 실행해
재생성합니다). 열 이름 규칙과 계산식은 저장소에 포함되는 `docs/feature_spec.json`에 기록합니다.
`timestamp`, `machine_status`, `label`, `state`를 앞에 붙입니다. 기간 분할 열은 넣지 않았으므로 모델 쪽에서
시각 기준으로 나눕니다.

In [ ]:
output_dir = project_root / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
parquet_path = output_dir / "pump_features.parquet"
table = pd.concat([status, label, state, features], axis=1).reset_index()
table.to_parquet(parquet_path, index=False)
reloaded = pd.read_parquet(parquet_path, columns=["timestamp", "label", f"{focus}__z{ZSCORE_WINDOW}"])
if len(reloaded) != len(table) or not reloaded["timestamp"].is_monotonic_increasing:
    raise ValueError("저장한 parquet을 다시 읽은 결과가 계산 결과와 다릅니다.")
print(f"저장 {parquet_path.relative_to(project_root).as_posix()} · {len(table):,}행 × {table.shape[1]:,}열 · "
      f"{parquet_path.stat().st_size / 2**20:,.0f} MB")

spec = {
    "generated_by": "notebooks/08_pump_rolling_features.ipynb",
    "source_rows": int(len(values)),
    "feature_count": int(features.shape[1]),
    "meta_columns": {
        "timestamp": "원본 시각", "machine_status": "원본 상태(특징에서 제외)",
        "label": "0 = NORMAL, 1 = BROKEN·RECOVERING",
        "state": f"normal / pre_failure(고장 전 {PRE_FAILURE_MINUTES}분 NORMAL) / broken / recovering",
    },
    "failure_starts": [start.strftime("%Y-%m-%d %H:%M:%S") for start in starts],
    "features": feature_spec(sensors, WINDOWS, lags=DIFF_LAGS, zscore_window=ZSCORE_WINDOW),
    "precursor_check": {
        "baseline_window": "고장 24~6시간 전", "recent_window": f"고장 전 {PRE_FAILURE_MINUTES}분",
        "level_threshold": LEVEL_THRESHOLD, "volatility_threshold": VOLATILITY_THRESHOLD,
        "top_sensors": precursor.head(10)["전조 건수(레벨 또는 변동)"].astype(int).to_dict(),
    },
    "zscore_rule": {
        "threshold": Z_THRESHOLD, "lead_hours": LEAD_HOURS,
        "top_sensors": rule.head(10)[["적중 건수(7건 중)", "정상 구간 오경보(%)"]].round(3)
        .rename(columns={"적중 건수(7건 중)": "hits_of_7", "정상 구간 오경보(%)": "false_alarm_pct"})
        .join(rule.head(10)["오경보 에피소드 수"].rename("false_alarm_episodes"))
        .to_dict(orient="index"),
    },
}
spec_path = project_root / "docs/feature_spec.json"
spec_path.write_text(json.dumps(spec, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"저장 {spec_path.relative_to(project_root).as_posix()}")

저장 data/processed/pump_features.parquet · 220,320행 × 886열 · 552 MB
저장 docs/feature_spec.json
